# Obravnava manjkajočih vrednosti — A: prikaz (telekom)
**Akademija UI za poslovne aplikacije — Modul 1, popoldanski sklop**

Ta zvezek poganja predavatelj. Vajo naredite v zvezku **`B_manjkajoce_vrednosti`** (spletna trgovina).

Cilj: znati **odkriti** manjkajoče podatke in izbrati **smiselno obravnavo**.

### Ponovitev
- **Kako manjka:** `NaN` (Pandas ga prepozna sam) ali **posebne oznake** (`"U"`, `"UNKW"`, `-1`, `999`, `1900-01-01`),
  ki jih moramo Pandasu povedati: `read_csv(na_values=...)`.
- **Manjkajoča vrednost ni ničla:** 0 nakupov je podatek, `NaN` pomeni, da vrednosti ne poznamo.
- Najprej **delež** (`df.isna().mean()`), nato **vzorec**: manjka naključno ali pri določenih strankah / v določenem obdobju?
- **Možnosti obravnave:**

| Metoda | Kdaj | Tveganje |
|---|---|---|
| Odstranitev vrstic | Manjka malo in naključno | Izgubimo podatke, pristranskost |
| Odstranitev stolpca | Manjka skoraj vse | Izgubimo informacijo |
| Mediana / povprečje | Numerični stolpec | Umetna konica, manjša razpršenost |
| Najpogostejša kategorija | Kategorični, manjka malo | Napihne največjo skupino |
| Oznaka »neznano« | Kategorični stolpec | Nova kategorija |
| Indikator manjkanja | Manjkanje nosi informacijo | En stolpec več |

- **Varovalka:** mediano in druge parametre izračunamo samo na **učnih** podatkih.
- **Univerzalno pravilne metode za nadomeščanje ni** — pomemben je razlog.

### Nastavitev okolja
Zaženite spodnjo celico (izberite jo in pritisnite **Shift + Enter**). Ničesar ne spreminjajte.
Celica deluje lokalno in v Google Colabu. Koda je skrita — odprete jo s klikom na celico.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:           # Google Colab: prenesemo gradivo z GitHuba
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not KOREN.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotnine>=0.15", "kagglehub"], check=True)
else:                                       # lokalno: poiščemo mapo, v kateri je utils/orodja.py
    KOREN = Path.cwd()
    while not (KOREN / "utils/orodja.py").exists() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
sys.path.insert(0, str(KOREN))              # da Python najde utils/orodja.py
print("Gradivo je v:", KOREN)

### Podatki
Funkcija `poskrbi_za_podatke()` iz `utils/orodja.py` ob prvem zagonu prenese podatke s Kaggla (~1 minuta)
in jih shrani v mapo `data/`. Kako to deluje, smo pokazali v zvezku `A_prvi_stik_s_podatki`.

In [ ]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()

### Knjižnice in videz grafov

In [ ]:
import numpy as np
import pandas as pd
from plotnine import *

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, BARVE_ODHOD
nastavi_slog()      # enoten videz grafov (koda je v zvezku A_vizualizacija)

---
# Del A — Prikaz: telekom
## A1. Koliko manjka?
`isna()` vrne `True`, kjer vrednost manjka. `.sum()` prešteje `True` vrednosti, `.mean()` da delež.

In [ ]:
telekom = pd.read_csv(DATA / "telekom.csv")
print(telekom.shape)

delez = (telekom.isna().mean().mul(100)               # % manjkajočih po stolpcih
         .rename_axis("stolpec").reset_index(name="odstotek"))
delez = delez[delez["odstotek"] > 0].sort_values("odstotek")
print(delez.sort_values("odstotek", ascending=False).round(2).to_string(index=False))

(ggplot(delez, aes(x="stolpec", y="odstotek"))
 + geom_col(fill=MODRA)
 + scale_x_discrete(limits=delez["stolpec"].tolist())
 + coord_flip()
 + labs(title="Delež manjkajočih vrednosti po stolpcih", x="", y="% vrstic"))

Na prvi pogled: `stevilo_avtomobilov` manjka pri polovici strank, `dohodek_razred` pri četrtini, ostalo je malo.
`zakonski_stan` manjka samo pri 1,7 % strank. Res?

## A2. Skrite manjkajoče vrednosti: posebne oznake
Pandas samodejno prepozna prazno celico, `NA`, `NaN`, `null` …, **ne pa** oznak, ki si jih je izmislil nekdo v podjetju.

In [ ]:
for stolpec in ["zakonski_stan", "splet_na_napravi", "dualband"]:
    print(stolpec, telekom[stolpec].value_counts(dropna=False).to_dict())

`U` = *unknown*, `UNKW` = *unknown*. To so manjkajoče vrednosti, preoblečene v kategorijo.
Pandasu to povemo že pri branju, s parametrom `na_values` (slovar: stolpec → seznam oznak):

In [ ]:
SKRITE_OZNAKE = {"zakonski_stan": ["U"], "splet_na_napravi": ["UNKW"], "dualband": ["U"]}

telekom = pd.read_csv(DATA / "telekom.csv", na_values=SKRITE_OZNAKE)
telekom.isna().mean().mul(100).sort_values(ascending=False).head(6).round(1)

`zakonski_stan` je skočil z 1,7 % na **39 %** manjkajočih.
Drugi pogosti primeri posebnih oznak: `-1`, `999`, `0`, `"?"`, `"N/A"`, `"ni podatka"`, datum `1900-01-01`.

## A3. Manjkajoča vrednost ni isto kot ničla
- `povp_minute = 0` → stranka **ni klicala**. To je podatek.
- `povp_minute = NaN` → podatka **nimamo**. Morda je klicala veliko.

In [ ]:
print("povp_minute = 0:  ", (telekom["povp_minute"] == 0).sum(), "strank")
print("povp_minute = NaN:", telekom["povp_minute"].isna().sum(), "strank")

Če bi `NaN` kar zamenjali z 0, bi 357 strankam, o katerih ne vemo nič, pripisali, da telefona sploh ne uporabljajo.

Včasih vrednost manjka **z dobrim razlogom**. Pri kom manjka povprečje minut zadnjih 6 mesecev?
Naredimo stolpec `True`/`False` in primerjamo skupini:

In [ ]:
telekom["manjka_6m"] = telekom["povp_minute_6m"].isna()

(ggplot(telekom, aes(x="manjka_6m", y="mesecev_narocnine"))
 + geom_boxplot(fill=MODRA, alpha=0.5)
 + labs(title="Mesecev naročnine: ali 6-mesečno povprečje manjka?",
        x="povp_minute_6m manjka", y="mesecev naročnine")).show()

telekom.groupby("manjka_6m")["mesecev_narocnine"].describe()[["count", "min", "50%", "max"]]

Manjka skoraj samo pri **novih strankah** (mediana 7 mesecev naročnine proti 17).
Šestmesečnega povprečja za stranko, ki je z nami pol leta, ne moremo dobro izračunati.

## A4. Vzorci: kje in pri kom manjka?
`isna()` vrne tabelo `True`/`False`. Narišemo jo kot mrežo: vsaka vrstica slike je ena stranka,
modro pomeni, da vrednost manjka. Za plotnine tabelo najprej pretvorimo v **dolgo obliko** (`melt`):
ena vrstica = ena celica slike (stranka, stolpec, manjka).

In [ ]:
stolpci = ["zakonski_stan", "dohodek_razred", "stevilo_avtomobilov", "splet_na_napravi", "povp_minute_6m", "cena_naprave"]
vzorec = telekom.sample(300, random_state=1).sort_values("zakonski_stan", na_position="first")

manjka = vzorec[stolpci].isna().reset_index(drop=True)     # tabela True/False
manjka["stranka"] = manjka.index                            # 0, 1, 2 … = vrstni red na sliki
dolga = manjka.melt(id_vars="stranka", var_name="stolpec", value_name="manjka")
print(dolga.head(3))

(ggplot(dolga, aes(x="stolpec", y="stranka", fill="manjka"))
 + geom_tile()
 + scale_fill_manual(values={False: "#f0efec", True: MODRA})
 + scale_x_discrete(limits=stolpci)
 + scale_y_reverse()                                        # prva stranka zgoraj
 + labs(title="Vzorec manjkajočih vrednosti (modro = manjka), 300 strank", x="", y="stranka")
 + theme(axis_text_x=element_text(rotation=20, ha="right")))

Modri bloki se pojavljajo **skupaj**: kjer manjka zakonski stan, manjkata tudi dohodek in število avtomobilov.
Preverimo s številkami: delež manjkajočega dohodka in števila avtomobilov **po zakonskem stanu**.
Manjkajoči zakonski stan poimenujemo `manjka`, da ostane kot svoja skupina.

In [ ]:
po_stanu = (telekom[["dohodek_razred", "stevilo_avtomobilov"]].isna()
            .groupby(telekom["zakonski_stan"].fillna("manjka")).mean().mul(100))
print(po_stanu.round(0))

dolga = po_stanu.reset_index().melt(id_vars="zakonski_stan", var_name="stolpec", value_name="odstotek")
(ggplot(dolga, aes(x="zakonski_stan", y="odstotek", fill="stolpec"))
 + geom_col(position="dodge")
 + scale_fill_manual(values=[MODRA, ORANZNA])
 + coord_flip()
 + labs(title="% manjkajočih glede na zakonski stan", x="zakonski stan", y="% manjkajočih", fill=""))

To niso naključne luknje. Verjetno za te stranke nimamo demografskih podatkov (npr. niso izpolnile obrazca).

Ali je manjkanje povezano z odhodom? Primerjajmo delež odhodov:

In [ ]:
for stolpec in ["dohodek_razred", "zakonski_stan", "povp_minute_6m"]:
    po_skupinah = telekom.groupby(telekom[stolpec].isna())["odhod"].mean()
    print(f"{stolpec:16s}  odhod, ko podatek je: {po_skupinah[False]:.1%}   ko manjka: {po_skupinah[True]:.1%}")

Pri `povp_minute_6m` je razlika velika (~ 40 % proti ~ 50 %): **že samo dejstvo, da podatek manjka, nekaj pove** o stranki.

## A5. Možnost 1: odstranimo vrstice
Najpreprostejše: `dropna()`. Brez argumentov odstrani vsako vrstico z vsaj eno manjkajočo vrednostjo,
s `subset=[...]` samo tiste, kjer manjka v izbranih stolpcih.

In [ ]:
print("Vse stranke:                     ", len(telekom))
print("Brez vrstic, kjer manjka dohodek:", len(telekom.dropna(subset=["dohodek_razred"])))
print("Brez vrstic, kjer manjka karkoli:", len(telekom.dropna()))

Če odstranimo vsako vrstico z vsaj eno manjkajočo vrednostjo, izgubimo **skoraj dve tretjini strank**.
Pa še nekaj: ne izgubimo jih naključno. Kdo ostane?

In [ ]:
pred = telekom["zakonski_stan"].fillna("manjka").value_counts(normalize=True)
po = telekom.dropna()["zakonski_stan"].fillna("manjka").value_counts(normalize=True)
primerjava = pd.DataFrame({"pred (vse stranke)": pred, "po dropna()": po}).fillna(0)
print(primerjava.round(3))

dolga = (primerjava.rename_axis("zakonski_stan").reset_index()
         .melt(id_vars="zakonski_stan", var_name="kdaj", value_name="delez"))
(ggplot(dolga, aes(x="zakonski_stan", y="delez", fill="kdaj"))
 + geom_col(position="dodge")
 + scale_fill_manual(values={"pred (vse stranke)": SIVA, "po dropna()": MODRA})
 + coord_flip()
 + labs(title="Zakonski stan: kdo ostane po dropna()?", x="zakonski stan", y="delež strank", fill=""))

Po `dropna()` ostanejo samo stranke z znanim zakonskim stanom. Vse ugotovitve in vsak model
bi veljali samo za **del** strank, o katerih vemo največ. To je pristranskost (*bias*).

Odstranitev je upravičena, kadar:
- manjka **malo** vrstic in manjkajo **naključno**, ali
- vrstica brez te vrednosti za naše vprašanje **nima smisla** (npr. ciljna spremenljivka manjka).

Stolpec, kjer manjka skoraj vse, pogosto raje odstranimo **v celoti**: `telekom.drop(columns=["stevilo_avtomobilov"])`.

## A6. Možnost 2: nadomestimo z mediano
`fillna(vrednost)` zamenja vse `NaN` z izbrano vrednostjo. Vse vrstice ostanejo. Kaj pa porazdelitev?
Narišemo histogram **po** nadomeščanju (modri stolpci) in **pred** njim (črn obris) — dve plasti z različnimi podatki.

In [ ]:
mediana = telekom["dohodek_razred"].median()
nadomesceno = telekom["dohodek_razred"].fillna(mediana)

print("Mediana:", mediana)
print(f"Std. odklon: {telekom['dohodek_razred'].std():.2f} → {nadomesceno.std():.2f}")
print(f"Delež strank v razredu 6: {(telekom['dohodek_razred'] == 6).mean():.0%} → {(nadomesceno == 6).mean():.0%}")

pred = pd.DataFrame({"dohodek_razred": telekom["dohodek_razred"].dropna()})
po = pd.DataFrame({"dohodek_razred": nadomesceno})
(ggplot(mapping=aes(x="dohodek_razred"))
 + geom_histogram(data=po, binwidth=1, fill=MODRA)                            # po: modri stolpci
 + geom_histogram(data=pred, binwidth=1, fill="none", color="black")          # pred: črn obris
 + scale_x_continuous(breaks=range(1, 10))
 + labs(title="dohodek_razred: nadomeščanje z mediano",
        subtitle="modro: po nadomeščanju, črn obris: pred", y="število strank"))

Vsem strankam brez podatka (četrtina vseh) smo pripisali razred 6 — delež razreda 6 je skočil z 19 % na 44 %.
Nastala je **umetna konica** in razpršenost se je zmanjšala.

Tak graf "pred in po" bomo potrebovali še v vaji, zato ga zapakiramo v **funkcijo** (kopija je v `utils/orodja.py`):

In [ ]:
def primerjaj_pred_po(pred, po, naslov=""):
    """Histogram po obravnavi (modri stolpci) in obris porazdelitve pred njo (črna črta)."""
    meje = np.histogram_bin_edges(pd.concat([pred, po]).dropna(), bins=40)
    return (ggplot(mapping=aes(x="vrednost"))
            + geom_histogram(data=pd.DataFrame({"vrednost": po.dropna()}), breaks=meje, fill=MODRA)
            + geom_histogram(data=pd.DataFrame({"vrednost": pred.dropna()}), breaks=meje,
                             fill="none", color="black")
            + labs(title=naslov, x="", y="Število vrstic",
                   subtitle=f"modro: po (n = {po.notna().sum()}), črn obris: pred (n = {pred.notna().sum()})"))

**Mediana ali povprečje?** Mediana je odporna na ekstreme. Pri dolgem repu (npr. `povp_minute`) je povprečje
potegnjeno navzgor, zato tam skoraj vedno izberemo mediano.

In [ ]:
print("povp_minute — povprečje:", round(telekom["povp_minute"].mean(), 1), "  mediana:", telekom["povp_minute"].median())

## A7. Kategorični stolpci: najpogostejša vrednost ali "neznano"?
`mode()` vrne najpogostejšo vrednost (seznam, zato `[0]`).
Obe možnosti narišemo kot **podgrafa** (`facet_wrap`): ena tabela, stolpec `metoda` pove, v kateri podgraf gre vrstica.

In [ ]:
najpogostejsa = telekom["zakonski_stan"].mode()[0]
print("Najpogostejša vrednost:", najpogostejsa)

z_najpogostejso = telekom["zakonski_stan"].fillna(najpogostejsa)
z_neznano = telekom["zakonski_stan"].fillna("neznano")

primerjava = pd.concat([
    z_najpogostejso.value_counts().reset_index().assign(metoda=f"fillna('{najpogostejsa}')"),
    z_neznano.value_counts().reset_index().assign(metoda="fillna('neznano')"),
])
(ggplot(primerjava, aes(x="zakonski_stan", y="count"))
 + geom_col(fill=MODRA)
 + coord_flip()
 + facet_wrap("~metoda")
 + labs(title="Zakonski stan po nadomeščanju", x="", y="število strank")
 + theme(figure_size=(10, 3.5)))

Ker manjka 39 %, bi z najpogostejšo vrednostjo (`M` = poročen) "poročili" še 39 % strank. Kategorija **`neznano`**
je tu poštenejša: ohrani informacijo, da podatka ni. Najpogostejšo vrednost uporabimo, ko manjka res malo.

## A8. Možnost 3: indikator manjkajoče vrednosti
Nadomestimo z mediano **in** dodamo nov stolpec, ki si zapomni, kje je manjkalo.
Tako model lahko uporabi informacijo iz A4 (nove stranke odhajajo redkeje).

In [ ]:
telekom["povp_minute_6m_manjka"] = telekom["povp_minute_6m"].isna().astype(int)      # 1 = manjkalo
telekom["povp_minute_6m"] = telekom["povp_minute_6m"].fillna(telekom["povp_minute_6m"].median())

telekom.loc[telekom["povp_minute_6m_manjka"] == 1, ["povp_minute_6m", "povp_minute_6m_manjka"]].head()

## A9. Varovalka: uhajanje podatkov (data leakage)
Ko gradimo model, podatke razdelimo na **učne** (model se uči) in **testne** (preverimo, kako dobro deluje na novih strankah).
Testni podatki morajo ostati "neznani" — tudi pri predprocesiranju.

**Mediano izračunamo samo na učnih podatkih** in isto številko uporabimo na testnih.
Podrobneje o delitvi v 3. sklopu.

In [ ]:
from sklearn.model_selection import train_test_split

telekom = pd.read_csv(DATA / "telekom.csv", na_values=SKRITE_OZNAKE)
ucni, testni = train_test_split(telekom, test_size=0.2, random_state=42)

mediana_ucni = ucni["dohodek_razred"].median()            # samo iz učnih podatkov!
ucni = ucni.fillna({"dohodek_razred": mediana_ucni})
testni = testni.fillna({"dohodek_razred": mediana_ucni})  # ista številka, ne nova mediana testnih

print("Mediana iz učnih podatkov:", mediana_ucni)

Isto naredi `SimpleImputer` iz scikit-learn: `fit` se uči samo na učnih, `transform` uporabimo na obeh.
`add_indicator=True` doda še indikator iz A8.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median", add_indicator=True)
imputer.fit(ucni[["povp_minute_6m"]])                     # nauči se samo na učnih
print("Naučena mediana:", imputer.statistics_)
imputer.transform(testni[["povp_minute_6m"]])[:5]          # stolpca: vrednost, indikator

---
Naprej: zvezek **`B_manjkajoce_vrednosti`** — manjkajoče vrednosti v spletni trgovini.